# yt2drive — YouTube to Google Drive

Tải video YouTube và lưu thẳng vào Google Drive.

**Các bước:**
1. Chạy Cell 1 — cài thư viện
2. Chạy Cell 2 — kết nối Google Drive
3. Sửa Cell 3 — điền URL và cấu hình
4. Chạy Cell 4 — bắt đầu tải


In [ ]:
# CELL 1: Cài đặt thư viện
!pip install -q yt-dlp
print('Cài xong yt-dlp.')

In [ ]:
# CELL 2: Kết nối Google Drive
from google.colab import drive
drive.mount('/content/drive')
print('Đã kết nối Google Drive.')

In [ ]:
# CELL 3: CẤU HÌNH — Chỉnh sửa tại đây

# URL video hoặc playlist YouTube
YOUTUBE_URL = 'https://www.youtube.com/watch?v=dQw4w9WgXcQ'

# Thư mục lưu trong Google Drive (tính từ My Drive)
# Ví dụ: 'Videos/YouTube' -> MyDrive/Videos/YouTube/
DRIVE_FOLDER = 'YouTube Downloads'

# Chất lượng video
# 'best'       -> chất lượng tốt nhất
# '1080'       -> Full HD
# '720'        -> HD
# '480'        -> SD
# '360'        -> SD thấp
# 'audio_only' -> chỉ âm thanh (MP3)
QUALITY = 'best'

# True = tải cả playlist | False = chỉ tải video đầu tiên
DOWNLOAD_PLAYLIST = False

# True  -> tên file = [Tên kênh] Tiêu đề.mp4
# False -> tên file = Tiêu đề.mp4
ADD_UPLOADER_TO_FILENAME = False

# ---
import os
SAVE_PATH = f'/content/drive/MyDrive/{DRIVE_FOLDER}'
os.makedirs(SAVE_PATH, exist_ok=True)

print(f'Thư mục lưu : {SAVE_PATH}')
print(f'Chất lượng  : {QUALITY}')
print(f'Playlist    : {DOWNLOAD_PLAYLIST}')
print('Cấu hình xong. Hãy chạy Cell 4.')

In [ ]:
# CELL 4: Bắt đầu tải
import yt_dlp
import os

def build_format(quality):
    if quality == 'audio_only':
        return 'bestaudio/best'
    elif quality == 'best':
        return 'bestvideo+bestaudio/best'
    else:
        return f'bestvideo[height<={quality}]+bestaudio/best[height<={quality}]'

if ADD_UPLOADER_TO_FILENAME:
    outtmpl = f'{SAVE_PATH}/[%(uploader)s] %(title)s.%(ext)s'
else:
    outtmpl = f'{SAVE_PATH}/%(title)s.%(ext)s'

def on_progress(d):
    if d['status'] == 'downloading':
        print(f"  {d['filename'].split('/')[-1]} — "
              f"{d.get('_percent_str','?'):>7} "
              f"({d.get('_speed_str','?'):>12}) "
              f"ETA {d.get('_eta_str','?')}")
    elif d['status'] == 'finished':
        print(f"  Xong: {d['filename'].split('/')[-1]}")

ydl_opts = {
    'format': build_format(QUALITY),
    'outtmpl': outtmpl,
    'noplaylist': not DOWNLOAD_PLAYLIST,
    'merge_output_format': 'mp4',
    'progress_hooks': [on_progress],
    'postprocessors': [{
        'key': 'FFmpegExtractAudio',
        'preferredcodec': 'mp3',
        'preferredquality': '192',
    }] if QUALITY == 'audio_only' else [],
    'quiet': True,
    'no_warnings': True,
}

print(f'Lấy thông tin: {YOUTUBE_URL}\n')

try:
    with yt_dlp.YoutubeDL({'quiet': True, 'no_warnings': True}) as ydl:
        info = ydl.extract_info(YOUTUBE_URL, download=False)

    if 'entries' in info:
        entries = list(info['entries'])
        count = len(entries) if DOWNLOAD_PLAYLIST else 1
        print(f'Playlist: "{info.get("title", "N/A")}"')
        print(f'Số video: {len(entries)} (sẽ tải: {count})\n')
    else:
        duration = info.get('duration', 0)
        mins, secs = divmod(duration, 60)
        print(f'Video     : {info.get("title", "N/A")}')
        print(f'Kênh      : {info.get("uploader", "N/A")}')
        print(f'Thời lượng: {mins}:{secs:02d}')
        print(f'Lượt xem  : {info.get("view_count", 0):,}\n')

    print(f'Lưu vào: {SAVE_PATH}')
    print('Bắt đầu tải...\n')
    print('-' * 60)

    with yt_dlp.YoutubeDL(ydl_opts) as ydl:
        ydl.download([YOUTUBE_URL])

    print('-' * 60)

    files = os.listdir(SAVE_PATH)
    print(f'\nHoàn thành! Các file trong "{DRIVE_FOLDER}":')
    for f in sorted(files):
        fpath = os.path.join(SAVE_PATH, f)
        size_mb = os.path.getsize(fpath) / (1024 * 1024)
        print(f'  {f} ({size_mb:.1f} MB)')

    print(f'\nMở Drive: https://drive.google.com/drive/folders/')

except yt_dlp.utils.DownloadError as e:
    print(f'Lỗi tải: {e}')
except Exception as e:
    print(f'Lỗi: {e}')
    raise

---

## Cell Bonus: Tải nhiều video cùng lúc

Điền danh sách URL rồi chạy cell dưới:

In [ ]:
# CELL BONUS: Tải nhiều video cùng lúc
import yt_dlp, os

URL_LIST = [
    'https://www.youtube.com/watch?v=VIDEO_ID_1',
    'https://www.youtube.com/watch?v=VIDEO_ID_2',
    'https://www.youtube.com/watch?v=VIDEO_ID_3',
]

DRIVE_FOLDER_BATCH = 'YouTube Downloads/Batch'
QUALITY_BATCH = '720'   # best / 1080 / 720 / 480 / audio_only

SAVE_PATH_BATCH = f'/content/drive/MyDrive/{DRIVE_FOLDER_BATCH}'
os.makedirs(SAVE_PATH_BATCH, exist_ok=True)

def build_format(q):
    if q == 'audio_only': return 'bestaudio/best'
    if q == 'best': return 'bestvideo+bestaudio/best'
    return f'bestvideo[height<={q}]+bestaudio/best[height<={q}]'

ydl_opts_batch = {
    'format': build_format(QUALITY_BATCH),
    'outtmpl': f'{SAVE_PATH_BATCH}/%(title)s.%(ext)s',
    'merge_output_format': 'mp4',
    'noplaylist': True,
    'quiet': False,
    'no_warnings': True,
    'postprocessors': [{
        'key': 'FFmpegExtractAudio',
        'preferredcodec': 'mp3',
    }] if QUALITY_BATCH == 'audio_only' else [],
}

print(f'Danh sách: {len(URL_LIST)} video')
print(f'Lưu vào  : {SAVE_PATH_BATCH}\n')

for i, url in enumerate(URL_LIST, 1):
    print(f'[{i}/{len(URL_LIST)}] Đang tải: {url}')
    try:
        with yt_dlp.YoutubeDL(ydl_opts_batch) as ydl:
            ydl.download([url])
        print(f'[{i}/{len(URL_LIST)}] Xong!\n')
    except Exception as e:
        print(f'[{i}/{len(URL_LIST)}] Loi: {e}\n')

print('Hoan thanh tat ca!')

---

## FAQ

| Câu hỏi | Trả lời |
|---|---|
| Video lỗi `403 Forbidden` | Bị chặn theo vùng hoặc cần đăng nhập. Thử URL khác. |
| Chất lượng `1080` không có | Video chưa có bản 1080p. Dùng `best` thay thế. |
| Không tìm thấy file trong Drive | Kiểm tra lại tên `DRIVE_FOLDER`. |
| Colab tự ngắt kết nối | F12 → Console → dán: `setInterval(() => document.querySelector('colab-toolbar-button#connect').click(), 60000)` |
